This notebook will be used to test two different scheduling heuristics.

In [37]:
import pandas as pd
import numpy as np

# System configuration: 4 conveyors, each with one shipping lane
N_CONVEYORS = 4

# 8 distinct shape types that the conveyor vision system can recognize
N_SHAPE_TYPES = 8
SHAPE_NAMES = ['cirle', 'pentagon', 'trapezoid', 'triangle', 'star', 'moon', 'heart', 'cross']

In [38]:
# Load the CSV files generated by the data generator notebook
# Each row is one order; columns are ragged (NaN-padded) since orders have varying numbers of item types
item_types_df = pd.read_csv('ranDataGen/order_itemtypes.csv', header=None)
quantities_df = pd.read_csv('ranDataGen/order_quantities.csv', header=None)

# Build a structured list of orders
# Each order becomes a list of (item_type_index, quantity) tuples
# Processing time for an order = total number of items (sum of quantities)
orders = []
processing_times = []

for i in range(len(item_types_df)):
    # Drop NaN values from ragged rows and convert to int
    types = item_types_df.iloc[i].dropna().astype(int).tolist()
    qtys = quantities_df.iloc[i].dropna().astype(int).tolist()
    order = list(zip(types, qtys))
    orders.append(order)
    # Processing time = total quantity of items in the order
    processing_times.append(sum(qtys))

# Print summary for verification
print(f"Number of orders: {len(orders)}")
print(f"Processing times: {processing_times}")
print(f"Total items: {sum(processing_times)}")
print()
for i, order in enumerate(orders):
    print(f"Order {i}: items={order}, processing_time={processing_times[i]}")

Number of orders: 5
Processing times: [5, 5, 7, 2, 2]
Total items: 21

Order 0: items=[(3, 3), (1, 2)], processing_time=5
Order 1: items=[(2, 3), (3, 1), (0, 1)], processing_time=5
Order 2: items=[(3, 3), (2, 3), (0, 1)], processing_time=7
Order 3: items=[(1, 1), (2, 1)], processing_time=2
Order 4: items=[(1, 2)], processing_time=2


In [39]:
# Parallel machine list-scheduling heuristic
# Treats each conveyor as a parallel machine and assigns orders one at a time
# to the least-loaded conveyor, balancing the workload across all conveyors
def schedule_orders(processing_times, n_conveyors, reverse=False):
    """
    Assign orders to conveyors using list scheduling.
    reverse=False: SPT (shortest processing time first)
    reverse=True: LPT (longest processing time first)
    """
    # Sort order indices by processing time (ascending for SPT, descending for LPT)
    sorted_indices = sorted(range(len(processing_times)),
                            key=lambda i: processing_times[i],
                            reverse=reverse)

    # Track the current load (total items assigned) on each conveyor
    loads = [0] * n_conveyors
    assignment = {c: [] for c in range(n_conveyors)}

    # Greedily assign each order to the conveyor with the smallest current load
    for order_idx in sorted_indices:
        # Find conveyor with minimum load (ties broken by lowest index)
        min_load = min(loads)
        conveyor = loads.index(min_load)
        assignment[conveyor].append(order_idx)
        loads[conveyor] += processing_times[order_idx]

    return assignment

In [40]:
# Build per-order rows with conveyor assignment (1-based)
# and write to CSV in the format expected by the FlexSim conveyor model
def build_output_df(assignment, orders, n_conveyors):
    """Build a DataFrame with one row per order, showing item counts and conveyor assignment."""
    rows = []
    for c in range(n_conveyors):
        for order_idx in assignment[c]:
            counts = [0] * N_SHAPE_TYPES
            for item_type, qty in orders[order_idx]:
                counts[item_type] += qty
            rows.append([c + 1] + counts)  # 1-based conveyor numbering

    columns = ['conv_num'] + SHAPE_NAMES
    return pd.DataFrame(rows, columns=columns)

def save_conveyor_csv(df, filepath):
    """Save CSV matching the example format: no-space header, space-after-comma data."""
    with open(filepath, 'w') as f:
        # Header row: comma-separated with no spaces (matches FlexSim import format)
        f.write(','.join(df.columns) + '\n')
        # Data rows: comma-space separated (matches example input format)
        for _, row in df.iterrows():
            f.write(', '.join(str(int(v)) for v in row) + '\n')
    print(f"Saved to {filepath}")

# Shortest Processing Time 

In [41]:
# Run SPT heuristic: sort orders from shortest to longest processing time,
# then assign each order to the least-loaded conveyor
spt_assignment = schedule_orders(processing_times, N_CONVEYORS, reverse=False)

# Display which orders were assigned to each conveyor and their total load
print("SPT (Shortest Processing Time) Assignment:")
for c in range(N_CONVEYORS):
    order_list = spt_assignment[c]
    load = sum(processing_times[i] for i in order_list)
    print(f"  Conveyor {c}: orders={order_list}, load={load}")

# Makespan = the maximum load across all conveyors (determines total completion time)
spt_makespan = max(sum(processing_times[i] for i in spt_assignment[c]) for c in range(N_CONVEYORS))
print(f"\nSPT Makespan: {spt_makespan}")

SPT (Shortest Processing Time) Assignment:
  Conveyor 0: orders=[3, 2], load=9
  Conveyor 1: orders=[4], load=2
  Conveyor 2: orders=[0], load=5
  Conveyor 3: orders=[1], load=5

SPT Makespan: 9


In [42]:
# Build the conveyor input table for SPT and save as CSV
# Each row represents one order with its shape counts and assigned conveyor (1-based)
spt_df = build_output_df(spt_assignment, orders, N_CONVEYORS)
print("SPT Conveyor Input:")
print(spt_df.to_string(index=False))
print()
save_conveyor_csv(spt_df, 'frConveryor/SPT_input.csv')

SPT Conveyor Input:
 conv_num  cirle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1      0         1          1         0     0     0      0      0
        1      1         0          3         3     0     0      0      0
        2      0         2          0         0     0     0      0      0
        3      0         2          0         3     0     0      0      0
        4      1         0          3         1     0     0      0      0

Saved to frConveryor/SPT_input.csv


# Longest Processing Time

In [43]:
# Run LPT heuristic: sort orders from longest to shortest processing time,
# then assign each order to the least-loaded conveyor
# LPT generally produces better (lower) makespans than SPT for parallel machine scheduling
lpt_assignment = schedule_orders(processing_times, N_CONVEYORS, reverse=True)

# Display which orders were assigned to each conveyor and their total load
print("LPT (Longest Processing Time) Assignment:")
for c in range(N_CONVEYORS):
    order_list = lpt_assignment[c]
    load = sum(processing_times[i] for i in order_list)
    print(f"  Conveyor {c}: orders={order_list}, load={load}")

# Makespan = the maximum load across all conveyors (determines total completion time)
lpt_makespan = max(sum(processing_times[i] for i in lpt_assignment[c]) for c in range(N_CONVEYORS))
print(f"\nLPT Makespan: {lpt_makespan}")

LPT (Longest Processing Time) Assignment:
  Conveyor 0: orders=[2], load=7
  Conveyor 1: orders=[0], load=5
  Conveyor 2: orders=[1], load=5
  Conveyor 3: orders=[3, 4], load=4

LPT Makespan: 7


In [44]:
# Build the conveyor input table for LPT and save as CSV
# Each row represents one order with its shape counts and assigned conveyor (1-based)
lpt_df = build_output_df(lpt_assignment, orders, N_CONVEYORS)
print("LPT Conveyor Input:")
print(lpt_df.to_string(index=False))
print()
save_conveyor_csv(lpt_df, 'frConveryor/LPT_input.csv')

LPT Conveyor Input:
 conv_num  cirle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1      1         0          3         3     0     0      0      0
        2      0         2          0         3     0     0      0      0
        3      1         0          3         1     0     0      0      0
        4      0         1          1         0     0     0      0      0
        4      0         2          0         0     0     0      0      0

Saved to frConveryor/LPT_input.csv


In [45]:
# Compare the two heuristics side-by-side
# Makespan is the key metric: it represents the time the last conveyor finishes
# A lower makespan means better load balancing across conveyors
print("=" * 50)
print("COMPARISON SUMMARY")
print("=" * 50)
print(f"{'Metric':<25} {'SPT':>10} {'LPT':>10}")
print("-" * 50)
print(f"{'Makespan':<25} {spt_makespan:>10} {lpt_makespan:>10}")
spt_total = sum(processing_times)
lpt_total = sum(processing_times)
print(f"{'Total items':<25} {spt_total:>10} {lpt_total:>10}")
print("-" * 50)
if lpt_makespan <= spt_makespan:
    print("LPT produces a better or equal makespan.")
else:
    print("SPT produces a better makespan.")

COMPARISON SUMMARY
Metric                           SPT        LPT
--------------------------------------------------
Makespan                           9          7
Total items                       21         21
--------------------------------------------------
LPT produces a better or equal makespan.
